In [1]:
import hail as hl
# Initialize Hail 
hl.init(default_reference = 'GRCh38')

Loading BokehJS ...

/opt/conda/miniconda3/lib/python3.10/site-packages/hail/context.py:352: UserWarning:

Using hl.init with a default_reference argument is deprecated. To set a default reference genome after initializing hail, call `hl.default_reference` with an argument to set the default reference genome.

/opt/conda/miniconda3/lib/python3.10/site-packages/hailtop/aiocloud/aiogoogle/user_config.py:43: UserWarning:

Reading spark-defaults.conf to determine GCS requester pays configuration. This is deprecated. Please use `hailctl config set gcs_requester_pays/project` and `hailctl config set gcs_requester_pays/buckets`.

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


SPARKMONITOR_LISTENER: Started SparkListener for Jupyter Notebook
SPARKMONITOR_LISTENER: Port obtained from environment: 56739
SPARKMONITOR_LISTENER: Application Started: application_1728056238285_0003 ...Start Time: 1728079324027


Running on Apache Spark version 3.3.2
SparkUI available at http://vep110-test-m.us-central1-a.c.daly-ibd.internal:38973
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.130-bea04d9c79b5
LOGGING: writing to /home/hail/hail-20241004-2202-0.2.130-bea04d9c79b5.log


In [2]:
mt = hl.read_matrix_table("gs://ibd-exomes-gnomad-subset/QC_round3/5.merge_moayeddi/merged.mt")
mt.count()

(22285775, 173330)

In [3]:
mt = mt.key_rows_by(**hl.min_rep(mt.locus, mt.alleles))

In [4]:
GGv4_1_ht = hl.read_table("gs://gcp-public-data--gnomad/release/4.1/ht/genomes/gnomad.genomes.v4.1.sites.ht/")

In [5]:
# # To improve the speed, extract the ht from the mt
# ht = mt.rows()
mt = mt.annotate_rows(gnomad_genomes_v4_1 = GGv4_1_ht[mt.row_key])
mt = mt.filter_rows(hl.is_defined(mt.gnomad_genomes_v4_1.filters) & (mt.gnomad_genomes_v4_1.filters.length() > 0), keep = True)
mt_gnomadv4_1_genome_filtered = mt.filter_rows(hl.is_defined(mt.gnomad_genomes_v4_1.filters) & (mt.gnomad_genomes_v4_1.filters.length() > 0), keep = False)

In [6]:
mt_gnomadv4_1_genome_filtered = hl.read_matrix_table('gs://ibd-exomes-gnomad-subset/QC_round3/6.final_variant_filter/gnomadv4.1_genome_filtered.mt')

In [7]:
mt_gnomadv4_1_genome_filtered.count()

(21990132, 173330)

In [3]:
GEv4_1_ht = hl.read_table("gs://gcp-public-data--gnomad/release/4.1/ht/exomes/gnomad.exomes.v4.1.sites.ht")

In [10]:
# filtered
mt_gnomadv4_1_genome_filtered = mt_gnomadv4_1_genome_filtered.annotate_rows(gnomad_exomes_v4_1 = GEv4_1_ht[mt_gnomadv4_1_genome_filtered.row_key])
mt_gnomadv4_1_filtered = mt_gnomadv4_1_genome_filtered.filter_rows(
                                                    hl.is_defined(mt_gnomadv4_1_genome_filtered.gnomad_exomes_v4_1.filters) 
                                                    & (mt_gnomadv4_1_genome_filtered.gnomad_exomes_v4_1.filters.length() > 0), keep = False)

In [11]:
mt_gnomadv4_1_filtered = mt_gnomadv4_1_filtered.checkpoint('gs://ibd-exomes-gnomad-subset/QC_round4/6.final_variant_filter/gnomadv4.1_genome_exome_filtered.mt', overwrite=True)

2024-07-18 22:29:23.930 Hail: INFO: wrote matrix table with 17991373 rows and 173330 columns in 42517 partitions to gs://ibd-exomes-gnomad-subset/QC_round4/6.final_variant_filter/gnomadv4.1_genome_exome_filtered.mt


In [11]:
mt_gnomadv4_1_filtered = hl.read_matrix_table('gs://ibd-exomes-gnomad-subset/QC_round4/6.final_variant_filter/gnomadv4.1_genome_exome_filtered.mt')

In [12]:
mt_gnomadv4_1_filtered.count()

(17991373, 173330)

In [13]:
mt_gnomadv4_1_filtered  = hl.variant_qc(mt_gnomadv4_1_filtered, name='variant_qc')
mt_gnomadv4_1_filtered_QCed = mt_gnomadv4_1_filtered.filter_rows(mt_gnomadv4_1_filtered.variant_qc.AC[1] > 0, keep = True)

In [14]:
mt_gnomadv4_1_filtered_QCed.checkpoint("gs://ibd-exomes-gnomad-subset/QC_round4/6.final_variant_filter/gnomadv4.1.filtered.QCed.final.mt", overwrite = True)

2024-07-19 16:00:50.954 Hail: INFO: wrote matrix table with 11213369 rows and 173330 columns in 42517 partitions to gs://ibd-exomes-gnomad-subset/QC_round4/6.final_variant_filter/gnomadv4.1.filtered.QCed.final.mt


In [16]:
mt_gnomadv4_1_filtered_QCed = mt_gnomadv4_1_filtered_QCed.annotate_cols(
                                    newSID = hl.delimit(mt_gnomadv4_1_filtered_QCed.s.split(' '), ","))
GT_gnomadv4_1_filtered_QCed = mt_gnomadv4_1_filtered_QCed.select_entries('GT')
hl.export_plink(GT_gnomadv4_1_filtered_QCed, 
                'gs://ibd-exomes-gnomad-subset/QC_round4/6.final_variant_filter/gnomadv4.1.QCed.GT_only.plink',
                ind_id = GT_gnomadv4_1_filtered_QCed.newSID)

Exception in thread "Thread-39" java.lang.NullPointerException062 + 64) / 42517]
	at sparkmonitor.listener.JupyterSparkMonitorListener$TaskUpdaterThread.$anonfun$run$1(CustomListener.scala:116)
	at scala.collection.TraversableLike$grouper$1$.apply(TraversableLike.scala:465)
	at scala.collection.TraversableLike$grouper$1$.apply(TraversableLike.scala:455)
	at scala.collection.mutable.ResizableArray.foreach(ResizableArray.scala:62)
	at scala.collection.mutable.ResizableArray.foreach$(ResizableArray.scala:55)
	at scala.collection.mutable.ArrayBuffer.foreach(ArrayBuffer.scala:49)
	at scala.collection.TraversableLike.groupBy(TraversableLike.scala:524)
	at scala.collection.TraversableLike.groupBy$(TraversableLike.scala:454)
	at scala.collection.AbstractTraversable.groupBy(Traversable.scala:108)
	at sparkmonitor.listener.JupyterSparkMonitorListener$TaskUpdaterThread.run(CustomListener.scala:116)
	at java.base/java.lang.Thread.run(Thread.java:829)
2024-07-19 17:38:11.282 Hail: INFO: merging 425

In [ ]:
hl.export_vcf(mt_gnomadv4_1_filtered_QCed.select_entries('GT'), 'gs://ibd-exomes-gnomad-subset/QC_round4/6.final_variant_filter/gnomadv4.1.QCed.GT_only.vcf.bgz')

2024-10-03 19:11:26.855 Hail: WARN: export_vcf: ignored the following fields:
    'a_index' (row)
    'was_split' (row)
    'nfe_ht' (row)
    'vep' (row)
    'gnomad_genomes_v4_1' (row)
    'gnomad_exomes_v4_1' (row)
    'variant_qc' (row)


In [2]:
mt_gnomadv4_1_filtered_QCed = hl.read_matrix_table("gs://ibd-exomes-gnomad-subset/QC_round4/6.final_variant_filter/gnomadv4.1.filtered.QCed.final.mt")

In [15]:
mt_gnomadv4_1_filtered_QCed.count()

(11213369, 173330)

In [4]:
mt_gnomadv4_1_filtered_QCed.describe()

----------------------------------------
Global fields:
    None
----------------------------------------
Column fields:
    's': str
----------------------------------------
Row fields:
    'locus': locus<GRCh38>
    'alleles': array<str>
    'rsid': str
    'info': struct {
        AC_raw: array<int32>, 
        AN_raw: int32, 
        AF_raw: array<float64>, 
        nhomalt_raw: array<int32>, 
        AC: array<int32>, 
        AN: int32, 
        AF: array<float64>, 
        nhomalt: array<int32>
    }
    'a_index': int32
    'was_split': bool
    'nfe_ht': struct {
        freq: array<struct {
            AC: int32, 
            AF: float64, 
            AN: int32, 
            homozygote_count: int32
        }>, 
        age_hist_het: array<struct {
            bin_edges: array<float64>, 
            bin_freq: array<int64>, 
            n_smaller: int64, 
            n_larger: int64
        }>, 
        age_hist_hom: array<struct {
            bin_edges: array<float64>, 
      

In [3]:
hl.export_vcf(mt_gnomadv4_1_filtered_QCed.select_entries('GT'), 'gs://ibd-exomes-gnomad-subset/QC_round4/6.final_variant_filter/GT_only.vcf.bgz')

2024-10-03 19:41:29.911 Hail: WARN: export_vcf: ignored the following fields:
    'a_index' (row)
    'was_split' (row)
    'nfe_ht' (row)
    'vep' (row)
    'gnomad_genomes_v4_1' (row)
    'gnomad_exomes_v4_1' (row)
    'variant_qc' (row)
2024-10-03 21:01:39.871 Hail: INFO: merging 42518 files totalling 30.3G...42517]
2024-10-03 21:09:52.480 Hail: INFO: while writing:
    gs://ibd-exomes-gnomad-subset/QC_round4/6.final_variant_filter/GT_only.vcf.bgz
  merge time: 8m12.6s


In [23]:
# mt_gnomadv4_1_v2_v3_filtered_QCed.write("gs://ibd-exomes-gnomad-subset/QC_round4/6.final_variant_filter/gnomadv4.1.v2.v3.QCed.final.mt", overwrite = True)

IOPub message rate exceeded.===================>           (32595 + 64) / 42517]
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_msg_rate_limit`.

Current values:
NotebookApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
NotebookApp.rate_limit_window=3.0 (secs)

2024-07-19 23:26:21.422 Hail: INFO: wrote matrix table with 10830192 rows and 173330 columns in 42517 partitions to gs://ibd-exomes-gnomad-subset/QC_round4/6.final_variant_filter/gnomadv4.1.v2.v3.QCed.final.mt


In [24]:
mt_gnomadv4_1_v2_v3_filtered_QCed = hl.read_matrix_table("gs://ibd-exomes-gnomad-subset/QC_round4/6.final_variant_filter/gnomadv4.1.v2.v3.QCed.final.mt")

In [25]:
mt_gnomadv4_1_v2_v3_filtered_QCed.count()

(10830192, 173330)